In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
import seaborn as sns
sns.set_style('whitegrid')
sns.set_palette("Set2")

# Leer los datos

In [28]:
df_fe = pd.read_csv("../../../data/respuestas_fede.csv")
print("Shape of data: ", df_fe.shape)

# globales
total = df_fe
marmol = df_fe.loc[df_fe["escuela"] == "Colegio Modelo Mármol"]
mantovani = df_fe.loc[df_fe["escuela"] == "Escuela Nueva Juan Mantovani"]
fig_name_prefix = 'info_en_wikipedia'
file_ext = '.png'
dpi_value = 200
include_title = True

Shape of data:  (369, 22)


# Misc por chique

In [43]:
# Sobre los artículos de Wikipedia
# data_total = df_fe
# data_marmol = marmol["sobre_wikipedia"].value_counts(normalize=True).sort_values(ascending=False)
# data_mantovani = mantovani["sobre_wikipedia"].value_counts(normalize=True).sort_values(ascending=False)

# dfs = [data_total, data_marmol, data_mantovani]
def analizar_misc_por_chique(df):
    donde_youtube = df["donde_youtube"]
    misc_youtube_por_chique = pd.get_dummies(donde_youtube[
        (donde_youtube != "En muchísimas computadoras (tantas que podrían llenar una cancha de fútbol)") &
        (donde_youtube != "No sé")
    ]).sum(axis=1)

    amiga_wpp = df["amiga_wpp"]
    misc_mando_foto_wpp = pd.get_dummies(amiga_wpp[
        (amiga_wpp != "Le mando una copia de mi foto.") &
        (amiga_wpp != "No sé")
    ]).sum(axis=1)

    amiga_no_ver_foto = df["amiga_no_ver_foto"]
    misc_amiga_no_ver_foto = pd.get_dummies(amiga_no_ver_foto[
        (amiga_no_ver_foto != "No puedo hacer nada, mi amiga ahora también tiene la foto y no tengo manera de sacársela.") &
        (amiga_no_ver_foto != "No sé")
    ]).sum(axis=1)

    como_viaja_wpp = df["como_viaja_wpp"]
    misc_como_viaja_wpp = pd.get_dummies(como_viaja_wpp[
        (como_viaja_wpp != "El mensaje se manda a través de una red de antenas y computadoras.") & 
        (como_viaja_wpp != "No sé.")
    ]).sum(axis=1)

    link_descarga = df["link_descarga"]
    misc_link_descarga = pd.get_dummies(link_descarga[
        (link_descarga != "Los mensajes recibidos no están verificados de ninguna forma. No podemos saber si el link es seguro.") & 
        (link_descarga != "No sé")
    ]).sum(axis=1)

    que_es_nube = df["que_es_nube"]
    misc_que_es_nube = pd.get_dummies(que_es_nube[
        (que_es_nube != "Muchísimas computadoras (tantas que podrían llenar una cancha de fútbol)") & 
        (que_es_nube != "No sé")
    ]).sum(axis=1)

    def analizar_misc(val):
        lista = val.split(",")
        if "PodemosutilizarlanubesinconexiónaInternet" in lista:
            return 1
        if "Sinlanubeseríaimposibleescucharmúsicaenelcelular" in lista:
            return 1
        if "Sinlanubenoseríaposiblehacerllamadasporcelular" in lista:
            return 1
        if "Sinlanubeseríaimposiblesacarfotosconelcelular" in lista:
            return 1
        return 0

    misc_afirmaciones_nube = df["afirmaciones_nube"].fillna("").str.replace(" ", "").apply(analizar_misc)

    como_se_comportan = df["como_se_comportan"]
    misc_como_se_comportan = pd.get_dummies(como_se_comportan[
        (como_se_comportan != "No sé.") &
        (como_se_comportan != "Internet y la nube tienen algunas funciones comunes y algunas funciones distintas.") &
        (como_se_comportan != "Con Internet podemos hacer más cosas que con la nube.")
    ]).sum(axis=1)

    sobre_wikipedia = df["sobre_wikipedia"]
    misc_sobre_wikipedia = pd.get_dummies(sobre_wikipedia[
        sobre_wikipedia != "Hace falta investigar más para decidir si la información es verdadera o falsa."
    ]).sum(axis=1)

    misc_totales_por_chique = pd.concat([misc_youtube_por_chique, 
                    misc_mando_foto_wpp, 
                    misc_amiga_no_ver_foto,
                    misc_como_viaja_wpp,
                    misc_link_descarga,
                    misc_que_es_nube,
                    misc_afirmaciones_nube,
                    misc_como_se_comportan,
                    misc_sobre_wikipedia], axis=1).sum(axis=1).value_counts().sort_index()
    return misc_totales_por_chique

mic_tot = analizar_misc_por_chique(total)
misc_marmol = analizar_misc_por_chique(marmol)
misc_manto = analizar_misc_por_chique(mantovani)

result = pd.concat([analizar_misc_por_chique(df) for df in [total, marmol, mantovani]], axis=1)

result = result.rename(columns={0: "Totales", 1: "Modelo Mármol", 2: "Juan Mantovani"})

print(result)



# result = pd.concat(dfs, axis=1, ignore_index=True)
# result = result.rename(columns={0: "Totales", 1: "Modelo Mármol", 2: "Juan Mantovani"},
#                        index={"Hace falta investigar más para decidir si la información es verdadera o falsa.": "Hay que\ninvestigar más",
#                               "Toda la información en el artículo es verdadera porque está verificada.": "Todo es\nverdad",
#                               "No hay forma de saber si la información del artículo es verdadera o falsa.": "Inverificable",
#                               "Todo lo que está en el artículo es falso: cualquier usuario pudo haberlo inventado.": "Todo es\nfalso"}).fillna(0)

# result = result.reindex(["Hay que\ninvestigar más","Todo es\nverdad","Inverificable","Todo es\nfalso"])

bar_width = 0.25

br1 = np.arange(len(result))
br2 = [x + bar_width for x in br1]
br3 = [x + bar_width for x in br2]

plt.bar(br1,result['Totales'],        width=bar_width, label = 'Totales')
plt.bar(br2,result['Modelo Mármol'],  width=bar_width, label = 'Modelo Mármol')
plt.bar(br3,result['Juan Mantovani'], width=bar_width, label = 'Juan Mantovani')

# plt.gca().yaxis.set_major_formatter(PercentFormatter(1))
# if include_title:
#        plt.title("Sobre los artículos de Wikipedia\nPor escuela", fontsize=15)
plt.xticks([x + bar_width for x in br1], result.index)
plt.legend()

# plt.savefig(fig_name_prefix + '_por_escuela' + file_ext, bbox_inches='tight', dpi=dpi_value)

     Totales  Modelo Mármol  Juan Mantovani
1.0        3            2.0               1
2.0       12            5.0               7
3.0       38           22.0              16
4.0       69           29.0              40
5.0       69           31.0              38
6.0       85           29.0              56
7.0       62           16.0              46
8.0       25           11.0              14
9.0        6            NaN               6
